<a href="https://colab.research.google.com/github/TienNguyen0712/hybrid-llm-tabular-pipeline-for-icu-mortality-prediction/blob/main/mimic_iv_rag_pipeline_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Trích xuất thông tin & So sánh hồ sơ tương đồng từ bộ MIMIC-IV dể giải quyết bài toán Dự đoán mức độ sinh tồn**

In [1]:
# ── CELL 1: Cài thư viện + Mount Drive ──────────────────────
# Bỏ comment nếu chạy trên Google Colab
from google.colab import drive
drive.mount('/content/drive')
!pip install tableone pyarrow seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import gc
import warnings
warnings.filterwarnings('ignore')

# Cài đặt style biểu đồ chuyên nghiệp
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")
print("✓ Libraries loaded")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
✓ Libraries loaded


## **Cấu hình đường dẫn**

In [2]:
# ── CELL 2: Đường dẫn dữ liệu ───────────────────────────────
# === CHỈNH Ở ĐÂY ===
# DATA_DIR = r"E:\KLTN\mimiciv\3.1\parquet"  # Windows local
DATA_DIR = "/content/drive/MyDrive/NCKH-DDU1231/mimic-iv-clinical-database-demo-2.2/mimic-iv-clinical-database-demo-2.2"  # Google Colab

import os
def load(folder, name):
    path = os.path.join(DATA_DIR, folder, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✓ {folder}/{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
        return df
    else:
        print(f"  ✗ {folder}/{name}: KHÔNG TÌM THẤY")
        return pd.DataFrame()

## **Nạp dữ liệu cơ bản**

### **Xây dựng bảng cohort dựa theo các điều kiện**

Xây dựng một bảng cohort được gộp từ bảng patients, admissions, icustay. Điều kiện gộp và lọc bao gồm:
- Các bệnh nhân phải là người trưởng thành (trên 18 tuổi)
- Lấy đợt năm ICU dầu tiên
- Thời gian nằm viện phải trên 24h kể từ khi vào ICU



In [3]:
# ── CELL 3: xây dựng cohort ───────────────────────────────
print("Loading core tables...")
# hosp
patients   = load("hosp", "patients.csv.gz")      # Thông tin bệnh nhân
admissions = load("hosp", "admissions.csv.gz")    # Thông tin nhập viện

# icu
icustays   = load("icu",  "icustays.csv.gz")      # Thông tin lần nằm ICU

# Merge thành cohort
cohort = (
    icustays
    .merge(admissions[["subject_id", "hadm_id", "admittime", "dischtime", "admission_type",
                       "admission_location", "edregtime", "edouttime", "deathtime",
                       "hospital_expire_flag", "insurance", "race"]],
           on=["subject_id", "hadm_id"], how="inner")
    .merge(patients[["subject_id", "gender", "anchor_age", "anchor_year", "dod"]],
           on="subject_id", how="inner")
)

initial_count = len(cohort)
print(f"Tổng số lượt ICU ban đầu: {initial_count:,}")

# Áp dụng các tiêu chí lọc

# Tiêu chí 1: Người trưởng thành (>= 18 tuổi)
cohort = cohort[cohort["anchor_age"] >= 18]
print(f"-> Sau khi lọc người trưởng thành (>=18t): {len(cohort):,} ca")

# Tiêu chí 2: Lấy ca ICU đầu tiên của mỗi bệnh nhân (First ICU stay per patient)
# Sắp xếp theo intime để chắc chắn lấy ca đầu tiên trong đời/lịch sử của bệnh nhân
cohort = (
    cohort.sort_values(["subject_id", "intime"])
          .groupby("subject_id", as_index=False)
          .first()
)
print(f"-> Sau khi chỉ lấy ca ICU đầu tiên của mỗi bệnh nhân: {len(cohort):,} ca")

# Tiêu chí 3: Loại những ca chết trong viện (hospital_expire_flag == 0)
cohort = cohort[cohort["hospital_expire_flag"] == 0]
print(f"-> Sau khi loại các ca tử vong trong viện: {len(cohort):,} ca")

# Tính thời gian nằm ICU (tính theo ngày)
cohort["icu_los_days"] = (
    pd.to_datetime(cohort["outtime"]) - pd.to_datetime(cohort["intime"])
).dt.total_seconds() / 86400

# Tiêu chí 4: ICU stay trên 1 ngày (> 1 ngày, tức > 24 giờ)
cohort = cohort[cohort["icu_los_days"] > 1.0]
print(f"-> Sau khi lọc ICU stay > 1 ngày: {len(cohort):,} ca")

# Tính các biến dẫn xuất bổ sung
cohort["icu_los_hours"] = cohort["icu_los_days"] * 24
cohort["hosp_los_days"] = (
    pd.to_datetime(cohort["dischtime"]) - pd.to_datetime(cohort["admittime"])
).dt.total_seconds() / 86400

cohort["age_group"] = pd.cut(
    cohort["anchor_age"],
    bins=[17, 30, 50, 65, 80, 120],
    labels=["18-30", "31-50", "51-65", "66-80", "80+"]
)

print("\n" + "="*50)
print(f"TẬP BỆNH NHÂN CUỐI CÙNG (FINAL COHORT):")
print(f"- Số lượng ca ICU / Bệnh nhân: {len(cohort):,}")
print(f"- Thời gian nằm ICU trung bình: {cohort['icu_los_days'].mean():.2f} ngày")
print(f"- Thời gian nằm viện trung bình: {cohort['hosp_los_days'].mean():.2f} ngày")
print("="*50)

Loading core tables...
  ✓ hosp/patients.csv.gz: 100 rows × 6 cols
  ✓ hosp/admissions.csv.gz: 275 rows × 16 cols
  ✓ icu/icustays.csv.gz: 140 rows × 8 cols
Tổng số lượt ICU ban đầu: 140
-> Sau khi lọc người trưởng thành (>=18t): 140 ca
-> Sau khi chỉ lấy ca ICU đầu tiên của mỗi bệnh nhân: 100 ca
-> Sau khi loại các ca tử vong trong viện: 89 ca
-> Sau khi lọc ICU stay > 1 ngày: 78 ca

TẬP BỆNH NHÂN CUỐI CÙNG (FINAL COHORT):
- Số lượng ca ICU / Bệnh nhân: 78
- Thời gian nằm ICU trung bình: 3.88 ngày
- Thời gian nằm viện trung bình: 9.82 ngày


In [4]:
cohort.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,insurance,race,gender,anchor_age,anchor_year,dod,icu_los_days,icu_los_hours,hosp_los_days,age_group
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,2157-11-18 22:56:00,2157-11-25 18:00:00,...,Other,WHITE,F,55,2157,None,1.118032,26.832778,6.794444,51-65
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,2110-04-11 15:08:00,2110-04-14 15:00:00,...,Other,WHITE,F,46,2110,None,1.338588,32.126111,2.994444,31-50
3,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2156-04-12 16:24:18,2156-04-17 15:57:08,4.981134,2156-04-12 14:16:00,2156-04-29 16:26:00,...,Medicare,WHITE,F,80,2155,None,4.981134,119.547222,17.090278,66-80
4,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,2141-05-22 20:17:00,2141-05-29 17:41:00,...,Medicare,UNKNOWN,M,81,2141,None,5.087512,122.100278,6.891667,80+
5,10002930,25696644,37049133,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2196-04-14 13:40:00,2196-04-15 16:54:44,1.135231,2196-04-14 12:25:00,2196-04-17 15:28:00,...,Medicare,BLACK/AFRICAN AMERICAN,F,48,2193,2201-12-24,1.135231,27.245556,3.127083,31-50


In [5]:
cohort.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'admittime', 'dischtime', 'admission_type',
       'admission_location', 'edregtime', 'edouttime', 'deathtime',
       'hospital_expire_flag', 'insurance', 'race', 'gender', 'anchor_age',
       'anchor_year', 'dod', 'icu_los_days', 'icu_los_hours', 'hosp_los_days',
       'age_group'],
      dtype='object')

### **Xây dựng bảng chart vitals từ bảng `chartevents`**


Để có thể xây dựng bảng này ta dựa vào đặc trưng có trong bảng `chartevents`. Các đặc trưng được lựa chọn:
- Nhịp tim - 220045
- Nhịp thở - 220210
- Nhiệt độ cơ thể độ C - 223762
- Nhiệt độ cơ thể độ F - 223761
- Huyết áp tâm thu/ tâm trương/ trung bình xâm lấn - 220050 / 220051 / 220052 (Trung bình)
- Huyết áp tâm thu/ tâm trương/ trung bình không xâm lấn - 220179 / 220180 / 220181
- SpO2 - 220277
- Glucose máu - 225664
- pH - 223830
- FiO2 - 223835
- Công thức máu -  220228 (Hemoglobin) / 220545 (Hematocrit)/ 220546 (WBC)227457 (Platelet)
- Chức năng thận - 220615 (Creatinine) / 225624 (BUN)
- Điện giải - 227442 (Potassium) / 220645 (Sodium) / 220602 (Chloride) / 227443 (HCO3) / 220635 (Magnesium)
- Khí máu - 223830 (pH Arterial) / 220224 (PaO2)/ 220235 (PaCO2) / 224828 (Base Excess) / 220227 (SaO2)
- Chức năng gan - 220644 (ALT) / 220587 (AST) / 225690 (Bilirubin Toàn phần)227456 (Albumin) / 225612 (Alkaline Phosphate)
- Thang điểm Glagsgow - 223900 (Verbal) / 223901 (Motor) / 220739 (Eye)
- Marker tim - 227429 (Troponin-T) / 227445 (CK-MB)227446 (BNP)
- Nội tiết - 228236 (Insulin) / 227463 (Cortisol)
- Cân nặng - 224639 (Cân nặng hàng ngày) / 226512 (Cân nặng nhập viện Kg)
- Chiều cao - 226707 / 226730

In [6]:
# ── CELL 4: xây dựng chart vitals ───────────────────────────────

# Định nghĩa mapping itemid
vital_itemids = {
    220045: "heart_rate", # Nhịp tim
    220179: "sbp_nibp",   #
    220180: "dbp_nibp",
    220181: "mbp_nibp",
    220050: "sbp_line",
    220051: "dbp_line",
    220052: "mbp_line",
    220210: "resp_rate",
    220277: "spo2",       # SpO2
    223761: "temp_c",     # Nhiệt độ c
    223762: "temp_f"      # Nhiệt độ F
}

print("Loading chartevents...")
chartevents   = load("icu",  "chartevents.csv.gz")

# Lọc chartevents theo Cohort và các itemid cần thiết
vitals_raw = chartevents[
    (chartevents["stay_id"].isin(cohort["stay_id"])) &
    (chartevents["itemid"].isin(vital_itemids.keys()))
][["subject_id", "hadm_id", "stay_id", "charttime", "itemid", "valuenum"]].copy()

# Gán tên chỉ số sinh tồn
vitals_raw["vital_name"] = vitals_raw["itemid"].map(vital_itemids)

# Làm sạch dữ liệu
def clean_vitals(df):
    df = df.copy()
    # Chuyển độ F sang độ C
    is_temp_f = df["vital_name"] == "temp_f"
    df.loc[is_temp_f, "valuenum"] = (df.loc[is_temp_f, "valuenum"] - 32) * 5 / 9
    df.loc[is_temp_f, "vital_name"] = "temp_c"

    # Loại bỏ giá trị phi thực tế
    df.loc[(df["vital_name"] == "heart_rate") & ((df["valuenum"] <= 0) | (df["valuenum"] > 300)), "valuenum"] = np.nan
    df.loc[(df["vital_name"] == "resp_rate") & ((df["valuenum"] <= 0) | (df["valuenum"] > 70)), "valuenum"] = np.nan
    df.loc[(df["vital_name"] == "spo2") & ((df["valuenum"] <= 0) | (df["valuenum"] > 100)), "valuenum"] = np.nan
    df.loc[(df["vital_name"] == "temp_c") & ((df["valuenum"] < 10) | (df["valuenum"] > 50)), "valuenum"] = np.nan

    # Lọc giá trị huyết áp phi thực tế (cả Line và NIBP)
    df.loc[(df["vital_name"].str.startswith("sbp")) & ((df["valuenum"] <= 0) | (df["valuenum"] > 400)), "valuenum"] = np.nan
    df.loc[(df["vital_name"].str.startswith("dbp")) & ((df["valuenum"] <= 0) | (df["valuenum"] > 300)), "valuenum"] = np.nan
    df.loc[(df["vital_name"].str.startswith("mbp")) & ((df["valuenum"] <= 0) | (df["valuenum"] > 300)), "valuenum"] = np.nan

    # Huyết áp gộp lại (ưu tiên xâm lấn - line, nếu không có lấy không xâm lấn - NIBP)
    return df.dropna(subset=["valuenum"])

vitals_cleaned = clean_vitals(vitals_raw)

# Kết hợp Huyết áp Systolic / Diastolic / Mean từ cả 2 nguồn
vitals_cleaned["vital_name"] = vitals_cleaned["vital_name"].replace({
    "sbp_line": "sbp", "sbp_nibp": "sbp",
    "dbp_line": "dbp", "dbp_nibp": "dbp",
    "mbp_line": "mbp", "mbp_nibp": "mbp"
})

# Pivot bảng sang dạng Wide format
vitals_pivoted = vitals_cleaned.pivot_table(
    index=["subject_id", "hadm_id", "stay_id", "charttime"],
    columns="vital_name",
    values="valuenum",
    aggfunc="mean"  # Trường hợp trong cùng 1 charttime có nhiều đo đạc
).reset_index()

vitals_pivoted.columns.name = None  # Xóa tên index cột

# Merge với cohort để lấy intime
# ── 5. Merge với cohort để lấy intime và lọc 24h đầu ────────────────────────
vitals_with_intime = vitals_pivoted.merge(
    cohort[["subject_id", "hadm_id", "stay_id", "intime"]], on=["subject_id", "hadm_id", "stay_id"], how="inner"
)

# Chuyển về Datetime và BỎ MÚI GIỜ (.dt.tz_localize(None)) để so sánh chuẩn xác
vitals_with_intime["charttime"] = pd.to_datetime(vitals_with_intime["charttime"]).dt.tz_localize(None)
vitals_with_intime["intime"] = pd.to_datetime(vitals_with_intime["intime"]).dt.tz_localize(None)

# Lọc trong khoảng [intime - 1 giờ, intime + 24 giờ]
# (Cho phép du di 1 giờ trước intime để tránh bỏ sót đo đạc lúc vừa tiếp nhận)
vitals_24h = vitals_with_intime[
    (vitals_with_intime["charttime"] >= vitals_with_intime["intime"] - pd.Timedelta(hours=1)) &
    (vitals_with_intime["charttime"] <= vitals_with_intime["intime"] + pd.Timedelta(hours=24))
].copy()

# ── 6. Gom nhóm tính Min, Max, Mean an toàn ─────────────────────────────────
agg_dict = {
    "heart_rate": ["min", "max", "mean"],
    "sbp": ["min", "max", "mean"],
    "dbp": ["min", "max", "mean"],
    "mbp": ["min", "max", "mean"],
    "resp_rate": ["min", "max", "mean"],
    "spo2": ["min", "max", "mean"],
    "temp_c": ["min", "max", "mean"]
}

# Đảm bảo tất cả các cột sinh tồn đều có mặt trong DataFrame trước khi agg
for col in agg_dict.keys():
    if col not in vitals_24h.columns:
        vitals_24h[col] = np.nan

# Kiểm tra dữ liệu trước khi thực hiện groupby
if len(vitals_24h) > 0:
    vitals_summary_24h = vitals_24h.groupby(["subject_id", "hadm_id", "stay_id"]).agg(agg_dict)
    # Phẳng hóa tên cột (ví dụ: heart_rate_min, sbp_mean...)
    vitals_summary_24h.columns = [f"{col[0]}_{col[1]}" for col in vitals_summary_24h.columns]
    vitals_summary_24h = vitals_summary_24h.reset_index()
else:
    print("⚠️ Cảnh báo: Không tìm thấy bản ghi sinh tồn nào trong 24h đầu!")
    cols = ["subject_id", "hadm_id", "stay_id"] + [f"{col}_{stat}" for col in agg_dict.keys() for stat in ["min", "max", "mean"]]
    vitals_summary_24h = pd.DataFrame(columns=cols)

print("="*50)
print(f"Thành công! Số ca ICU có sinh tồn 24h: {len(vitals_summary_24h):,}")
print("="*50)

Loading chartevents...
  ✓ icu/chartevents.csv.gz: 668,862 rows × 11 cols
Thành công! Số ca ICU có sinh tồn 24h: 78


In [ ]:
vitals_summary_24h.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'heart_rate_min', 'heart_rate_max',
       'heart_rate_mean', 'sbp_min', 'sbp_max', 'sbp_mean', 'dbp_min',
       'dbp_max', 'dbp_mean', 'mbp_min', 'mbp_max', 'mbp_mean',
       'resp_rate_min', 'resp_rate_max', 'resp_rate_mean', 'spo2_min',
       'spo2_max', 'spo2_mean', 'temp_c_min', 'temp_c_max', 'temp_c_mean'],
      dtype='object')

### **Xây dựng bảng lab tests từ bảng `labevents`**


Để có thể xây dựng bảng này ta dựa vào đặc trưng có trong bảng `chartevents`. Các đặc trưng được lựa chọn:
- Nhiệt độ cơ thể - 50825
- SpO2 / Bão hòa O2 - 50817
- Glucose máu - 50931 (Huyết tương )/ 50809 (Máu toàn phần)
- pH - 50820
- Công thức máu -  51221 (Hematocrit) / 51222 (Hemoglobin) / 51301 (Bạch cầu - WBC) / 51265 (Tiểu cầu) / 51279 (Hồng cầu - RBC)
- Chức năng thận - 50912 (Creatinine) / 51006 (BUN)
- Điện giải - 50971 (Kali / Potassium) / 50983 (Natri / Sodium) / 50902 (Clo / Chloride) / 50882 (Bicarbonate / HCO3) / 50893 (Calci toàn phần) / 50808 (Calci tự do) / 50960 (Magie) / 50970 (Phosphate)
- Khí máu - 50821 (pO2) / 50818 (pCO2) / 50802 (Base Excess) / 50820 (pH) / 50868 (Anion Gap)
- Chức năng gan - 50861 (ALT / SGPT) / 50878 (AST / SGOT) / 50863 (Alkaline Phosphatase) / 50885 (Bilirubin toàn phần) / 50862 (Albumin) / 50976 (Protein toàn phần)
- Marker tim - 51003 (Troponin T) / 50911 (CK-MB) / 50910 (CK tổng) / 50963 (NT-proBNP)
- Nội tiết - 50993 (TSH) / 50995 (Free T4) / 51001 (T3) / 50909 (Cortisol) / 50965 (PTH)

In [ ]:
labevents = load("hosp", "labevents.csv.gz")


In [9]:
# hosp
prescriptions   = load("hosp",  "prescriptions.csv.gz")

# icu
procedureevents   = load("icu",  "procedureevents.csv.gz")
inputevents    = load("icu",  "inputevents.csv.gz")
outputevents   = load("icu",  "outputevents.csv.gz")

  ✓ hosp/prescriptions.csv.gz: 18,087 rows × 21 cols
  ✓ icu/procedureevents.csv.gz: 1,468 rows × 22 cols
  ✓ icu/inputevents.csv.gz: 20,404 rows × 26 cols
  ✓ icu/outputevents.csv.gz: 9,362 rows × 9 cols
